In [15]:
import pandas as pd
import pyarrow.parquet as pq
import numpy as np

from sklearn.svm import SVC
from sklearn.metrics import classification_report
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# 1.0 - Classificação de Presença Baseada em Fatores Socioeconômicos Usando Redes Neurais

In [2]:
colunas = ['Q001','Q002','Q003', 'Q004', 'Q005', 'Q006', 'Q007', 'Q008', 'Q009', 'Q010', 'Q011', 'Q012', 'Q013', 'Q014', 'Q015', 'Q016', 'Q017', 'Q018', 
           'Q019', 'Q020', 'Q021', 'Q022', 'Q023', 'Q024', 'Q025', 'TP_PRESENCA_LC', 'TP_PRESENCA_CH', 'TP_PRESENCA_CN', 'TP_PRESENCA_MT', 
           'TP_FAIXA_ETARIA', 'TP_SEXO','TP_ESTADO_CIVIL', 'TP_COR_RACA', 'TP_ESCOLA', 'TP_ST_CONCLUSAO', 'IN_TREINEIRO', 
           'NU_ANO', 'TP_LOCALIZACAO_ESC','TP_SIT_FUNC_ESC', 'NU_NOTA_CN', 'NU_NOTA_CH', 'NU_NOTA_LC', 'NU_NOTA_MT', 'NU_NOTA_REDACAO', 'TP_DEPENDENCIA_ADM_ESC']

df = pd.read_parquet("enem_parquet", columns = colunas)

## 1.1 - Criando uma Amostra

In [4]:
df = df.sample(n=50_000, random_state=42)

## 1.2- Limpando os Dados

In [5]:
FALTOU = (
    (df['TP_PRESENCA_CH'] != 1) | 
    (df['TP_PRESENCA_LC'] != 1) | 
    (df['TP_PRESENCA_CN'] != 1) | 
    (df['TP_PRESENCA_MT'] != 1)
)

df['FALTOU'] = FALTOU.astype(int)

In [6]:
df = df[df['FALTOU'] == 0]

In [7]:
df = df[df['TP_ESCOLA'].isin([2,3])]
df = df[df['TP_ESTADO_CIVIL'].isin([1,2,3,4])]

In [8]:
df['TP_LOCALIZACAO_ESC'] = df['TP_LOCALIZACAO_ESC'].fillna(0)
df['TP_DEPENDENCIA_ADM_ESC'] = df['TP_DEPENDENCIA_ADM_ESC'].fillna(0)
df['TP_SIT_FUNC_ESC'] = df['TP_SIT_FUNC_ESC'].fillna(0)

In [9]:
questionario = [f'Q{str(i).zfill(3)}' for i in range(1, 26) if i != 5]
df = df.dropna(subset=questionario)

## 1.3 - Criando Rótulo

In [10]:
df['MEDIA'] = (df['NU_NOTA_CN'] + df['NU_NOTA_CH'] + df['NU_NOTA_MT']+  df['NU_NOTA_LC'] + df['NU_NOTA_REDACAO']) / 5

df['CLASSE'] = df.groupby('NU_ANO')['MEDIA'].transform(
    lambda x: pd.qcut(x, q=2, labels=[0,1])
).astype(int)
    
df['CLASSE'] = df['CLASSE'].astype(int)

## 1.4 - Tratando os Dados

In [11]:
nominais = ['TP_ESCOLA', 'TP_DEPENDENCIA_ADM_ESC',
            'TP_ESTADO_CIVIL', 'TP_LOCALIZACAO_ESC', 'TP_SIT_FUNC_ESC']

ordinais = ['TP_FAIXA_ETARIA', 'TP_ST_CONCLUSAO']

categorias_quest = []
for i in range(1, 26):
    if i == 5:
        continue
    elif i == 6:
        categorias_quest.append(list('ABCDEFGHIJKLMNOPQ'))  
    elif i == 25:
        categorias_quest.append(list('AB'))                  
    elif i in [1, 2]:
        categorias_quest.append(list('ABCDEFGH'))            
    elif i in [3, 4]:
        categorias_quest.append(list('ABCDEF'))              
    else:
        categorias_quest.append(list('ABCDE'))               

binarias = ['IN_TREINEIRO']

In [16]:
pipe_nominal = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

pipe_ordinal = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)

pipe_quest = OrdinalEncoder(
    categories=categorias_quest,
    handle_unknown='use_encoded_value',
    unknown_value=-1
)

pipe_continua = MinMaxScaler()

preprocessor = ColumnTransformer(transformers=[
    ('nominal',      pipe_nominal,  nominais),
    ('ordinal',      pipe_ordinal,  ordinais),
    ('questionario', pipe_quest,    questionario),
    ('binaria',      'passthrough', binarias),
], remainder='drop')


## 1.5- Construção da Matriz X e Vetor y

In [17]:
X = preprocessor.fit_transform(df[colunas])
X = X.astype(np.float32)
y = df['CLASSE']

## 1.6 - Separação em Dados de Treino e Teste

In [18]:
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
x_train, x_val, y_train, y_val = train_test_split(x_train, y_train, test_size=0.2, random_state=42)

In [22]:
pipe = Pipeline([
    ('svm', SVC(kernel='rbf', class_weight='balanced', random_state=42))
])

params = {
    'svm__C': [0.1, 1, 10],
    'svm__gamma': ['scale', 'auto']
}

grid = GridSearchCV(pipe, params, cv=5, scoring='f1', n_jobs=-1, verbose=1)
grid.fit(x_train, y_train)  

print(grid.best_params_)

Fitting 5 folds for each of 6 candidates, totalling 30 fits
{'svm__C': 10, 'svm__gamma': 'auto'}


In [23]:
print(classification_report(y_val, grid.predict(x_val)))

              precision    recall  f1-score   support

           0       0.67      0.77      0.72       853
           1       0.75      0.65      0.69       899

    accuracy                           0.71      1752
   macro avg       0.71      0.71      0.71      1752
weighted avg       0.71      0.71      0.71      1752

